In [0]:
%sql
-- ============================================================
-- 04) EXTRAS PRO (SIN REPETIR TABLAS YA CREADAS EN NOTEBOOK 03)
-- Requiere que ya existan:
--   - asbdevsuperstore.silver_schema.superstore_silver
--   - asbdevsuperstore.gold_schema.sales_by_region_month
--   - (y las otras gold que ya creaste)
-- ============================================================

USE CATALOG asbdevsuperstore;

In [0]:
%sql
-- ============================================================
-- A) CLONES (Shallow / Deep) sobre una tabla GOLD existente
-- ============================================================
-- SHALLOW CLONE: copia "metadata", NO duplica datos (rápido y barato)
CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.sales_by_region_month_shallow
SHALLOW CLONE asbdevsuperstore.gold_schema.sales_by_region_month
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/sales_by_region_month_shallow';

-- DEEP CLONE: duplica físicamente los datos (más caro, útil para backup)
CREATE TABLE IF NOT EXISTS asbdevsuperstore.gold_schema.sales_by_region_month_deep
DEEP CLONE asbdevsuperstore.gold_schema.sales_by_region_month
LOCATION 'abfss://superstoredata@adsldevsuperstore.dfs.core.windows.net/gold/sales_by_region_month_deep';

-- Ver detalles para entender qué se creó
DESCRIBE DETAIL asbdevsuperstore.gold_schema.sales_by_region_month_shallow;
DESCRIBE DETAIL asbdevsuperstore.gold_schema.sales_by_region_month_deep;

In [0]:
%sql
-- ============================================================
-- B) PERFORMANCE: OPTIMIZE + ZORDER (mejor data skipping)
-- ============================================================
-- OPTIMIZE: compacta archivos pequeños
-- ZORDER: mejora consultas cuando filtras por esas columnas
OPTIMIZE asbdevsuperstore.gold_schema.sales_by_region_month
ZORDER BY (region, order_month);

In [0]:
%sql
-- ============================================================
-- C) LIQUID CLUSTERING (si tu workspace lo soporta)
-- ============================================================
-- Si te da error de feature/sintaxis, comenta este bloque.
ALTER TABLE asbdevsuperstore.gold_schema.sales_by_region_month
CLUSTER BY (region, order_month);

In [0]:
%sql
-- ============================================================
-- D) CACHING (acelerar lecturas repetidas en la sesión)
-- ============================================================
CACHE TABLE asbdevsuperstore.gold_schema.sales_by_region_month;
-- Para limpiar cache:
-- UNCACHE TABLE asbdevsuperstore.gold_schema.sales_by_region_month;

In [0]:
%sql
-- ============================================================
-- E) CDF (Change Data Feed) en SILVER (para CDC)
-- ============================================================
-- Activa CDF para capturar cambios (INSERT/UPDATE/DELETE/MERGE)
ALTER TABLE asbdevsuperstore.silver_schema.superstore_silver
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Para ver versiones y poder leer cambios
DESCRIBE HISTORY asbdevsuperstore.silver_schema.superstore_silver;

-- Ejemplo (si quieres probar):
-- SELECT * FROM table_changes('asbdevsuperstore.silver_schema.superstore_silver', 0);

In [0]:
%sql
-- ============================================================
-- F) MERGE (UPSERT) ejemplo simple sobre SILVER
-- ============================================================
-- Creamos un staging temporal con 2 filas reales
CREATE OR REPLACE TEMP VIEW stg_updates AS
SELECT
  row_id,
  sales
FROM asbdevsuperstore.silver_schema.superstore_silver
WHERE row_id IS NOT NULL
LIMIT 2;

-- Simulamos cambios: sales + 10
CREATE OR REPLACE TEMP VIEW stg_updates_changed AS
SELECT
  row_id,
  sales + 10 AS sales
FROM stg_updates;

-- MERGE: actualiza si existe
MERGE INTO asbdevsuperstore.silver_schema.superstore_silver t
USING stg_updates_changed s
ON t.row_id = s.row_id
WHEN MATCHED THEN UPDATE SET
  t.sales = s.sales;

In [0]:
%sql
-- ============================================================
-- G) VACUUM (limpieza de archivos antiguos)
-- ============================================================
-- Limpia archivos que ya no se necesitan por versiones viejas.
-- OJO: respeta la retención por defecto (seguro para labs).
VACUUM asbdevsuperstore.gold_schema.sales_by_region_month;

In [0]:
%sql
-- ============================================================
-- H) GOVERNANCE / METADATA (linaje se ve mejor en el UI)
-- ============================================================
-- Permisos asignados
SHOW GRANTS ON TABLE asbdevsuperstore.silver_schema.superstore_silver;

-- Metadata extendida (owner, properties, etc.)
DESCRIBE EXTENDED asbdevsuperstore.silver_schema.superstore_silver;

-- Detalle (location, formato, etc.)
DESCRIBE DETAIL asbdevsuperstore.silver_schema.superstore_silver;

-- Linaje: Catalog Explorer -> Lineage (UI)